<a href="https://colab.research.google.com/github/gift-framework/GIFT/blob/main/docs/colab_k3_cap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# 1. SETUP
# ================================================================
import subprocess, sys, os, time, json, hashlib, zipfile, math
from datetime import datetime, timezone
from itertools import combinations_with_replacement

T_SESSION = time.time()
TIMESTAMP = datetime.now(tz=timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = f'/content/k3_cap_{TIMESTAMP}'
os.makedirs(OUT_DIR, exist_ok=True)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'mpmath'])

import numpy as np
import torch
import mpmath
mpmath.mp.dps = 50
iv = mpmath.iv

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.complex128
rdtype = torch.float64

print(f'Timestamp: {TIMESTAMP}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print(f'Output: {OUT_DIR}')

Timestamp: 20260416T125431Z
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.1 GB
PyTorch: 2.10.0+cu128
Output: /content/k3_cap_20260416T125431Z


In [2]:
# ================================================================
# 2. MONTE CARLO SAMPLING on FERMAT QUARTIC K3
# ================================================================
# X: f(z) = z0^4 + z1^4 + z2^4 + z3^4 = 0  in  CP^3
# We work in the patch z0 = 1 (affine coords w1, w2, w3 with constraint)

N_POINTS = 5000  # sample points on K3
torch.manual_seed(42)

def sample_k3_points(N):
    """Sample N random points on the Fermat quartic K3.

    Strategy: in patch z3=1, the constraint becomes
    z0^4 + z1^4 + z2^4 + 1 = 0
    Given random z0, z1, solve for z2^4 = -1 - z0^4 - z1^4, take any 4th root.
    """
    # Sample z0, z1 uniformly in a complex ball
    z0 = (torch.randn(N, dtype=rdtype, device=device) +
          1j * torch.randn(N, dtype=rdtype, device=device)).to(dtype) * 0.8
    z1 = (torch.randn(N, dtype=rdtype, device=device) +
          1j * torch.randn(N, dtype=rdtype, device=device)).to(dtype) * 0.8

    # z2^4 = -1 - z0^4 - z1^4
    c = -1 - z0**4 - z1**4
    # z2 = c^(1/4): principal 4th root
    z2 = c ** (0.25)
    # z3 = 1 by choice of patch
    z3 = torch.ones(N, dtype=dtype, device=device)

    # Homogeneous coordinates
    Z = torch.stack([z0, z1, z2, z3], dim=1)  # (N, 4)

    # Verify constraint
    f = (Z**4).sum(dim=1)
    constraint_residual = f.abs().max().item()
    print(f'  Sampled {N} K3 points')
    print(f'  Max constraint residual |f(z)| = {constraint_residual:.2e}')

    return Z

Z = sample_k3_points(N_POINTS)
print(f'  Z shape: {Z.shape}')
print(f'  Z dtype: {Z.dtype}')

  Sampled 5000 K3 points
  Max constraint residual |f(z)| = 1.07e-13
  Z shape: torch.Size([5000, 4])
  Z dtype: torch.complex128


In [3]:
# ================================================================
# 3. DONALDSON SECTIONS: monomials of degree k on CP^3
# ================================================================
# Basis: all monomials z0^a z1^b z2^c z3^d with a+b+c+d = k
# Number of sections: C(k+3, 3)

K_DEGREE = 4  # Donaldson degree

# Generate multi-indices (a, b, c, d) with a+b+c+d = k
def generate_monomials(k, n_vars=4):
    """Return list of tuples (a_0, ..., a_{n-1}) with sum = k."""
    if n_vars == 1:
        return [(k,)]
    result = []
    for i in range(k + 1):
        for rest in generate_monomials(k - i, n_vars - 1):
            result.append((i,) + rest)
    return result

monomials = generate_monomials(K_DEGREE, 4)
N_SECT = len(monomials)
print(f'k = {K_DEGREE}, N_sections = {N_SECT}')

# Store monomial exponents as a tensor
mono_exp = torch.tensor(monomials, dtype=torch.long, device=device)  # (N_SECT, 4)

def evaluate_sections(Z, mono_exp):
    """Evaluate s_alpha(z) = z0^a0 * z1^a1 * z2^a2 * z3^a3 at points Z.

    Args:
        Z: (N, 4) complex points
        mono_exp: (N_SECT, 4) integer exponents
    Returns:
        S: (N, N_SECT) complex matrix
    """
    # Z: (N, 4), mono_exp: (N_SECT, 4)
    # S[n, alpha] = prod_i Z[n, i]^mono_exp[alpha, i]
    # Use broadcasting: (N, 1, 4) ** (1, N_SECT, 4) = (N, N_SECT, 4)
    # Then reduce product over i
    powers = Z.unsqueeze(1) ** mono_exp.unsqueeze(0)  # (N, N_SECT, 4)
    S = powers.prod(dim=2)  # (N, N_SECT)
    return S

S = evaluate_sections(Z, mono_exp)
print(f'Sections S: {S.shape}, dtype={S.dtype}')
print(f'  |S|^2 min/max/mean: {(S.abs()**2).min().item():.2e} / {(S.abs()**2).max().item():.2e} / {(S.abs()**2).mean().item():.2e}')

k = 4, N_sections = 35
Sections S: torch.Size([5000, 35]), dtype=torch.complex128
  |S|^2 min/max/mean: 2.35e-18 / 5.68e+04 / 2.58e+01


In [4]:
# ================================================================
# 4. KÄHLER POTENTIAL and RICCI via AUTOGRAD
# ================================================================
# Kähler potential on ambient P^3 (Donaldson form):
#   K(z, z̄) = (1/k) * log(Σ H_{αβ} s_α(z) s̄_β(z̄))
#
# Metric (restricted to K3):
#   g_{i j̄} = ∂²K / (∂z^i ∂z̄^j)
#
# Ricci:
#   R_{i j̄} = -∂²log det(g) / (∂z^i ∂z̄^j)
#
# For Ricci-flat: we want R_{i j̄} = λ g_{i j̄} with λ=0 (or Einstein with Λ=0)
#
# For projective CY: the Ricci-flat condition is
#   det(g_{i j̄}) = |Ω|² (mass-monge-ampere)
# where Ω is the holomorphic volume form.

def kahler_potential(Z, H, mono_exp, k=4):
    """Donaldson Kähler potential at points Z.

    K(z) = (1/k) * log( Σ_{α,β} H[α,β] s_α(z) conj(s_β(z)) )

    Args:
        Z: (N, 4) complex points (requires_grad for derivatives)
        H: (N_SECT, N_SECT) Hermitian matrix
        mono_exp: (N_SECT, 4) exponents
        k: degree
    Returns:
        K: (N,) real tensor
    """
    S = evaluate_sections(Z, mono_exp)  # (N, N_SECT)
    # Σ_{α,β} H[α,β] s_α conj(s_β) = S @ H @ conj(S).T (diagonal)
    # = Σ_α Σ_β H[α,β] S[n,α] conj(S[n,β])
    quadform = torch.einsum('na,ab,nb->n', S, H, S.conj())
    # Should be real positive
    K = torch.log(quadform.real) / k
    return K

def compute_induced_metric(Z, H, mono_exp, k=4):
    """Compute g_{i j̄} = ∂²K/∂z^i∂z̄^j at each K3 point.

    Uses complex-step differentiation via autograd.
    Returns pullback to tangent space of K3 at each point.
    """
    # For now: compute the ambient Hessian, then project onto K3 tangent
    N = Z.shape[0]
    Z = Z.detach().requires_grad_(True)

    # Compute K at each point, then Hessian
    # PyTorch autograd works for complex tensors
    K_vals = kahler_potential(Z, H, mono_exp, k)  # (N,) real

    # First derivative: dK/dz^i
    # We want ∂K/∂z̄^j (Wirtinger): use autograd on complex Z
    # For Kähler: ∂²K/(∂z^i ∂z̄^j) = g_{i j̄}

    # Approach: gradient of K w.r.t. Z (real), then split into holo/antiholo parts
    K_sum = K_vals.sum()
    grad_K = torch.autograd.grad(K_sum, Z, create_graph=True)[0]  # (N, 4) complex

    # grad_K[n, i] = ∂K/∂z^i + ∂K/∂z̄^i (Wirtinger convention)
    # For Kähler real K: grad_K = 2 Re(∂K/∂z) in each component
    # Actually for complex autograd in PyTorch, grad gives the Wirtinger derivative

    # For the Hessian, we take another gradient
    # This is expensive: Jacobian of grad_K w.r.t. Z

    # SIMPLER: compute g analytically from H and S derivatives
    return grad_K

# Quick test: evaluate K at a few points
H_init = torch.eye(N_SECT, dtype=dtype, device=device)  # identity = Fubini-Study pullback

with torch.no_grad():
    K_test = kahler_potential(Z[:10], H_init, mono_exp, K_DEGREE)
    print(f'K at first 10 points: {K_test.cpu().numpy()}')
    print(f'K range: [{K_test.min().item():.4f}, {K_test.max().item():.4f}]')

print('\nKähler potential evaluation OK.')

K at first 10 points: [0.71895929 1.70500058 0.71332544 1.63681053 1.6346216  0.93576805
 2.40265023 0.82688905 1.3527939  1.18210836]
K range: [0.7133, 2.4027]

Kähler potential evaluation OK.


In [5]:
# ================================================================
# 5. SIMPLIFIED MONGE-AMPÈRE RESIDUAL
# ================================================================
# For a projective CY surface X ⊂ P^n, the Ricci-flat condition is
# the Monge-Ampère equation:
#   det(g_{i j̄}) = |Ω|²
# where Ω is the holomorphic top form restricted to X.
#
# For the Fermat quartic:
#   Ω|_X = Res[(dz0 ∧ dz1 ∧ dz2 ∧ dz3) / df]
#
# We measure the Ricci-flatness via the "σ-measure":
#   σ² = (1/Vol) ∫_X (det(g) / |Ω|² - c)² dμ
# where c normalizes the integral to 0.
#
# For this prototype, we compute g at sample points via analytical
# derivatives of K, then measure the Monge-Ampère deviation.

def compute_metric_analytical(Z, H, mono_exp, k=4):
    """Compute g_{i j̄} analytically on the AMBIENT P^3.

    For K(z) = (1/k) log(φ) with φ = S H S†, we have:
    ∂K/∂z^i = (1/k) (∂φ/∂z^i) / φ
    ∂²K/(∂z^i ∂z̄^j) = (1/k) [(∂²φ/∂z^i∂z̄^j)/φ - (∂φ/∂z^i)(∂φ/∂z̄^j)/φ²]

    This is the Fubini-Study-like induced metric on P^3, restricted to K3.
    """
    N = Z.shape[0]
    S = evaluate_sections(Z, mono_exp)  # (N, N_SECT)

    # φ = S @ H @ S†  at each point
    phi = torch.einsum('na,ab,nb->n', S, H, S.conj()).real  # (N,)

    # ∂s_α/∂z^i = a_i * z^(a - e_i), i.e., derivative of monomial
    # (N, N_SECT, 4): partial derivatives of each section w.r.t. each z^i
    # s_α(z) = prod_j z_j^a_j
    # ∂s_α/∂z^i = a_i * z_i^(a_i - 1) * prod_{j≠i} z_j^a_j
    #           = a_i * s_α(z) / z_i  (when a_i > 0)

    # Build ∂S/∂z^i: (N, N_SECT, 4)
    # For each α, i: coeff = mono_exp[α, i], power reduced by 1 in direction i

    # Reduced exponents: mono_exp_reduced[α, i] = mono_exp[α] - e_i (could be negative)
    # Use: ∂s_α/∂z^i = mono_exp[α,i] * z^mono_exp / z^e_i if mono_exp[α,i] > 0

    # Faster: ∂s_α/∂z^i = mono_exp[α,i] * s_α(z) / z_i
    # But this has /z issues when z_i = 0. Use explicit computation:
    a = mono_exp.float()  # (N_SECT, 4)
    # For each i, compute s_α^{(i)} = a_{α,i} * (z_i^{a_{α,i}-1}) * prod_{j≠i} z_j^{a_{α,j}}
    # This is a * S / z_i, handled via masking

    # Simpler approach: use autograd but vectorized
    Z_grad = Z.detach().clone().requires_grad_(True)

    def phi_fn(Z_):
        S_ = evaluate_sections(Z_, mono_exp)
        return torch.einsum('na,ab,nb->n', S_, H, S_.conj()).real

    # Compute ∂φ/∂z and ∂²φ/(∂z∂z̄) using autograd
    phi_vals = phi_fn(Z_grad)

    # Gradient: (N, 4) complex — this is ∂φ/∂z̄ (Wirtinger) * 2
    # Actually for holomorphic φ in z, the gradient w.r.t. complex Z gives ∂φ/∂z̄
    # For φ Hermitian (φ = S H S†), ∂φ/∂z = Σ_β H_{αβ} (∂s_α/∂z) conj(s_β)

    # This is getting complex — let's just return the quadratic form and
    # defer Ricci computation to a follow-up notebook

    return phi, S

phi, S = compute_metric_analytical(Z, H_init, mono_exp, K_DEGREE)

print(f'Monge-Ampère quadratic form φ:')
print(f'  min = {phi.min().item():.6e}')
print(f'  max = {phi.max().item():.6e}')
print(f'  mean = {phi.mean().item():.6e}')
print(f'  all positive: {(phi > 0).all().item()}')

# Measure σ deviation from Ricci-flat (simplified)
# For identity H (Fubini-Study), K3 is NOT Ricci-flat
# σ = std(log φ) / mean(log φ) is a rough deviation measure
log_phi = torch.log(phi)
sigma_FS = (log_phi.std() / log_phi.abs().mean()).item()
print(f'\nSimplified σ (Fubini-Study baseline): {sigma_FS:.6f}')
print('(Higher values = further from Ricci-flat; Donaldson optimization reduces this)')

Monge-Ampère quadratic form φ:
  min = 5.130994e+00
  max = 4.469537e+05
  mean = 9.043800e+02
  all positive: True

Simplified σ (Fubini-Study baseline): 0.430489
(Higher values = further from Ricci-flat; Donaldson optimization reduces this)


In [6]:
# ================================================================
# 6. KÄHLER METRIC via REAL AUTOGRAD (corrected)
# ================================================================
# PyTorch's complex autograd uses Wirtinger conventions that are confusing.
# Safer: split z = x + i*y (real X, Y), compute real Hessian, assemble g.
#
# For K(x, y) real, the Kähler metric is:
#   g_{i j̄} = (1/4) [(∂²K/∂x^i∂x^j + ∂²K/∂y^i∂y^j)
#                  + i (∂²K/∂y^i∂x^j - ∂²K/∂x^i∂y^j)]
#
# This IS Hermitian by construction.

def kahler_potential_real(X, Y, H, mono_exp, k=4):
    """K(x, y) as real function of real (X, Y)."""
    Z = (X + 1j * Y).to(dtype)  # (N, 4) complex
    S = evaluate_sections(Z, mono_exp)
    phi = torch.einsum('na,ab,nb->n', S, H, S.conj()).real
    return torch.log(phi) / k  # (N,)

def kahler_metric_real(Z, H, mono_exp, k=4):
    """Compute Kähler metric g_{i j̄} via real autograd.

    Returns: g (N, 4, 4) complex, Hermitian, positive-definite on P^3.
    """
    N = Z.shape[0]
    X = Z.real.detach().clone().requires_grad_(True)  # (N, 4)
    Y = Z.imag.detach().clone().requires_grad_(True)

    # Vectorized Hessian using functorch or manual double-grad
    # Since we want per-point Hessian, use torch.func.vmap for efficiency
    from torch.func import jacrev, vmap

    def K_single(x, y):
        """K at a single point (x, y): shape (4,) each → scalar."""
        z = (x + 1j*y).to(dtype)  # (4,) complex
        s = torch.stack([torch.prod(z**mono_exp[a].to(dtype)) for a in range(N_SECT)])  # (N_SECT,)
        phi = (s.conj() @ H @ s).real
        return torch.log(phi) / k

    # Hessians: ∂²K/∂x_i∂x_j, ∂²K/∂y_i∂y_j, ∂²K/∂x_i∂y_j
    K_xx = vmap(jacrev(jacrev(K_single, argnums=0), argnums=0))(X, Y)  # (N, 4, 4)
    K_yy = vmap(jacrev(jacrev(K_single, argnums=1), argnums=1))(X, Y)  # (N, 4, 4)
    K_xy = vmap(jacrev(jacrev(K_single, argnums=0), argnums=1))(X, Y)  # (N, 4, 4)
    # K_yx = K_xy^T (symmetric mixed partials)
    K_yx = K_xy.transpose(1, 2)

    # Assemble Kähler metric: g_{i j̄} = (1/4)[(K_xx + K_yy) + i(K_yx - K_xy)]
    g_re = 0.25 * (K_xx + K_yy)      # (N, 4, 4) real
    g_im = 0.25 * (K_yx - K_xy)      # (N, 4, 4) real
    g = g_re + 1j * g_im             # (N, 4, 4) complex

    return g

# Test with identity H (Fubini-Study on P^3)
print('Computing Kähler metric via REAL autograd (Wirtinger-safe)...')
t0 = time.time()
N_test = 50
g_test = kahler_metric_real(Z[:N_test], H_init, mono_exp, K_DEGREE)
print(f'  Time: {time.time()-t0:.2f}s for {N_test} points')

# Hermiticity check
hermiticity = (g_test - g_test.conj().transpose(1,2)).abs().max().item()
print(f'  Hermiticity ||g - g†||_max = {hermiticity:.2e}')

# Eigenvalues (should all be positive for a Kähler metric)
with torch.no_grad():
    eigs = torch.linalg.eigvalsh(g_test)  # real eigenvalues for Hermitian
    print(f'  Eigenvalue range: [{eigs.min().item():.4e}, {eigs.max().item():.4e}]')
    print(f'  All positive: {(eigs > 0).all().item()}')

det_g = torch.linalg.det(g_test).abs()
print(f'  |det(g)| range: [{det_g.min().item():.4e}, {det_g.max().item():.4e}]')

if hermiticity < 1e-8 and (eigs > 0).all():
    print('\n  ✓ Kähler metric correctly computed (P^3 ambient)')
else:
    print('\n  ✗ Still issues — debugging needed')

Computing Kähler metric via REAL autograd (Wirtinger-safe)...
  Time: 10.93s for 50 points
  Hermiticity ||g - g†||_max = 2.22e-16
  Eigenvalue range: [-5.5613e-16, 9.5333e-01]
  All positive: False
  |det(g)| range: [2.6056e-21, 3.7865e-17]

  ✗ Still issues — debugging needed


In [7]:
# ================================================================
# 6b. DEBUG — Why det(g) ≈ 0: projective rank analysis
# ================================================================
# Expected: g on P^3 has rank 3 (not 4). The radial direction z → λz
# is a null direction for the Fubini-Study metric.
# On K3 ⊂ P^3 (dim 2 complex), we need to pull back to K3 tangent (rank 2).

print('=== DEBUG: Eigenvalue analysis ===\n')

with torch.no_grad():
    eigs_all = torch.linalg.eigvalsh(g_test)  # (N, 4) sorted ascending

    # Show eigenvalue distribution at first 5 points
    print('Eigenvalues at first 5 points (4 per point, sorted):')
    for n in range(5):
        print(f'  Point {n}: {eigs_all[n].cpu().numpy()}')

    # Statistics
    print(f'\nGlobal stats over {g_test.shape[0]} points:')
    for r in range(4):
        col = eigs_all[:, r]
        print(f'  λ_{r}: min={col.min().item():.3e}, max={col.max().item():.3e}, '
              f'median={col.median().item():.3e}')

    # Count zero vs nonzero eigenvalues (threshold 1e-8)
    zeros = (eigs_all.abs() < 1e-8).sum(dim=1)
    print(f'\n  Points with ≥1 zero eigenvalue (|λ|<1e-8): {(zeros > 0).sum().item()}/{g_test.shape[0]}')
    print(f'  Typical rank: {4 - zeros.float().mean().item():.1f}')

    # Check if null direction is radial (z^i direction)
    # For P^n with Fubini-Study: ∂K/∂log(r²) = 1/2, so radial grad ∝ z̄^i
    # The null eigenvector should be proportional to z
    print(f'\n=== Null eigenvector analysis ===')
    for n in range(3):
        g_n = g_test[n]
        evals, evecs = torch.linalg.eigh(g_n)
        # Smallest eigenvalue (the null one)
        null_vec = evecs[:, 0]
        z_n = Z[n]
        # Overlap with z (radial direction)
        overlap = (null_vec.conj() @ z_n).abs() / (null_vec.abs().norm() * z_n.abs().norm())
        print(f'  Point {n}: λ_0={evals[0].item():.3e}, '
              f'|<null_vec, z>|/||.||||.|| = {overlap.item():.4f}')

print('\n=== Interpretation ===')
print('Rank-3 metric on P^3 is EXPECTED:')
print('  - 4 complex coords on P^3, but 1 direction is the radial scaling z → λz')
print('  - Fubini-Study is degenerate in this direction (homogeneous of degree 0)')
print('  - True tangent space at [z] ∈ P^3 is 3-dimensional complex')
print('  - For K3 ⊂ P^3: pullback gives rank 2 (dim_C K3 = 2)')
print('\nNext step: pull back g to K3 tangent space, then Ricci.')

=== DEBUG: Eigenvalue analysis ===

Eigenvalues at first 5 points (4 per point, sorted):
  Point 0: [-3.12539981e-17  4.35369848e-01  4.47351415e-01  5.24349820e-01]
  Point 1: [6.33052201e-17 1.01781265e-01 1.60237553e-01 2.20356011e-01]
  Point 2: [-6.84854865e-17  2.21364341e-01  4.88947717e-01  5.95062467e-01]
  Point 3: [-2.22963062e-17  7.33956773e-02  1.10929437e-01  2.74127176e-01]
  Point 4: [-1.43110043e-16  1.11263979e-01  1.49517460e-01  2.50358430e-01]

Global stats over 50 points:
  λ_0: min=-5.561e-16, max=5.170e-16, median=7.566e-18
  λ_1: min=3.594e-02, max=4.354e-01, median=1.752e-01
  λ_2: min=4.140e-02, max=5.981e-01, median=2.957e-01
  λ_3: min=1.185e-01, max=9.533e-01, median=4.305e-01

  Points with ≥1 zero eigenvalue (|λ|<1e-8): 50/50
  Typical rank: 3.0

=== Null eigenvector analysis ===
  Point 0: λ_0=-3.125e-17, |<null_vec, z>|/||.||||.|| = 1.0000
  Point 1: λ_0=6.331e-17, |<null_vec, z>|/||.||||.|| = 1.0000
  Point 2: λ_0=-6.849e-17, |<null_vec, z>|/||.||||.

In [8]:
# ================================================================
# 7. PULLBACK to K3 TANGENT SPACE
# ================================================================
# T_z K3 = { v ∈ C^4 : v ⊥ z (projective) AND df(z)·v = 0 (surface) }
#
# For f = Σ z_i^4:  df/dz_i = 4 z_i^3
#
# Build orthonormal basis of T_z K3 via Gram-Schmidt on the 2D complement.
# Then pull back g_{P^3} to this basis: g_K3 = P† g_P3 P  where P is (4, 2).

def compute_k3_tangent_basis(Z):
    """For each point z ∈ K3, compute a (4, 2) complex orthonormal basis of T_z K3.

    T_z K3 is the intersection of:
      - z⊥ (projective: v · z̄ = 0)
      - ker(df): v · ∇f̄(z̄) = 0  where ∇f = (4 z_i^3)

    Uses Gram-Schmidt on the 4×4 projector onto these constraints.
    """
    N = Z.shape[0]
    # Normal direction 1: radial = z (up to normalization)
    n1 = Z / Z.abs().norm(dim=1, keepdim=True)  # (N, 4) unit

    # Normal direction 2: grad f = 4 z^3 (element-wise cube)
    grad_f = 4.0 * (Z ** 3)  # (N, 4)
    # Orthogonalize grad_f against n1
    proj = torch.einsum('na,na->n', grad_f, n1.conj()).unsqueeze(1) * n1
    n2_raw = grad_f - proj
    n2 = n2_raw / (n2_raw.abs().norm(dim=1, keepdim=True) + 1e-30)

    # Now find 2 vectors orthogonal to both n1 and n2
    # Use e_0, e_1, e_2, e_3 standard basis, pick 2 that give nonzero projections
    # Gram-Schmidt: take e_0 - <e_0, n1> n1 - <e_0, n2> n2, etc.

    # Build (N, 4, 4) projector P = I - n1 n1† - n2 n2†
    I = torch.eye(4, dtype=dtype, device=device).unsqueeze(0).expand(N, 4, 4)
    P1 = torch.einsum('na,nb->nab', n1, n1.conj())
    P2 = torch.einsum('na,nb->nab', n2, n2.conj())
    P = I - P1 - P2  # (N, 4, 4) projector onto T_z K3 (in ambient C^4)

    # P has rank 2. Extract orthonormal basis via SVD.
    # Equivalently, compute eigendecomposition of P (Hermitian)
    # The 2 eigenvectors with eigenvalue 1 span T_z K3.
    evals, evecs = torch.linalg.eigh(P)  # (N, 4), (N, 4, 4), sorted ascending
    # Take the 2 largest eigenvalues (should be 1, 1)
    basis = evecs[:, :, -2:]  # (N, 4, 2)

    return basis, evals[:, -2:]

# Test
print('Computing K3 tangent basis...')
basis, evals = compute_k3_tangent_basis(Z[:N_test])
print(f'  Basis shape: {basis.shape}')
print(f'  Non-trivial eigenvalues (should be ≈1): mean={evals.mean().item():.6f}, '
      f'range=[{evals.min().item():.6f}, {evals.max().item():.6f}]')

# Verify: basis vectors are orthogonal to z and to grad_f
with torch.no_grad():
    # basis[n, :, 0], basis[n, :, 1] span T_z K3
    # Check basis[n,:,0] · z_n = 0  and  basis[n,:,0] · grad_f_n = 0
    z_dot = torch.einsum('nab,na->nb', basis, Z[:N_test].conj()).abs()  # (N, 2)
    grad_f = 4.0 * (Z[:N_test] ** 3)
    gf_dot = torch.einsum('nab,na->nb', basis, grad_f.conj()).abs()  # (N, 2)
    print(f'  Max |<basis, z>| = {z_dot.max().item():.2e}  (should be ~0)')
    print(f'  Max |<basis, grad_f>| = {gf_dot.max().item():.2e}  (should be ~0)')

    # Basis orthonormality
    gram = torch.einsum('nab,nac->nbc', basis, basis.conj())  # (N, 2, 2)
    I2 = torch.eye(2, dtype=dtype, device=device).unsqueeze(0)
    print(f'  Max |basis† basis - I| = {(gram - I2).abs().max().item():.2e}')

# Pull back g_P^3 to K3 tangent: g_K3 = basis† g_P3 basis
# g_K3 is (N, 2, 2) complex Hermitian, should be positive-definite
g_K3 = torch.einsum('nai,nab,nbj->nij', basis.conj(), g_test, basis)  # (N, 2, 2)

print(f'\n=== K3 PULLBACK RESULTS ===')
print(f'g_K3 shape: {g_K3.shape}')

# Hermiticity
herm_K3 = (g_K3 - g_K3.conj().transpose(1,2)).abs().max().item()
print(f'  Hermiticity: {herm_K3:.2e}')

# Eigenvalues (both should be positive now)
with torch.no_grad():
    eigs_K3 = torch.linalg.eigvalsh(g_K3)  # (N, 2)
    print(f'  Eigenvalue range: [{eigs_K3.min().item():.4e}, {eigs_K3.max().item():.4e}]')
    print(f'  All positive: {(eigs_K3 > 0).all().item()}')

det_K3 = torch.linalg.det(g_K3).real
print(f'  det(g_K3) range: [{det_K3.min().item():.4e}, {det_K3.max().item():.4e}]')
print(f'  log det range: [{torch.log(det_K3.abs()).min().item():.4f}, '
      f'{torch.log(det_K3.abs()).max().item():.4f}]')

if herm_K3 < 1e-10 and (eigs_K3 > 0).all():
    print('\n  ✓ K3 Kähler metric correctly computed (rank 2, positive definite)')
else:
    print('\n  ✗ Issue in pullback')

Computing K3 tangent basis...
  Basis shape: torch.Size([50, 4, 2])
  Non-trivial eigenvalues (should be ≈1): mean=1.000000, range=[1.000000, 1.000000]
  Max |<basis, z>| = 7.36e-16  (should be ~0)
  Max |<basis, grad_f>| = 1.72e-14  (should be ~0)
  Max |basis† basis - I| = 1.33e-15

=== K3 PULLBACK RESULTS ===
g_K3 shape: torch.Size([50, 2, 2])
  Hermiticity: 9.58e-17
  Eigenvalue range: [3.5951e-02, 6.8887e-01]
  All positive: True
  det(g_K3) range: [1.8960e-03, 2.2038e-01]
  log det range: [-6.2680, -1.5124]

  ✓ K3 Kähler metric correctly computed (rank 2, positive definite)


In [9]:
# ================================================================
# 8. MONGE-AMPÈRE LOSS + HOLOMORPHIC VOLUME FORM
# ================================================================
# For a CY surface X ⊂ P^n, Ricci-flat Kähler condition:
#   det(g_K3) = c · |Ω|²
# where Ω is the holomorphic (2,0)-form on K3.
#
# For X = {f = 0}:  Ω = Res[ (dz_0 ∧ dz_1 ∧ dz_2 ∧ dz_3) / df ]
# At a point z ∈ K3, Ω restricted to T_z K3 depends on which coordinate
# is used as local affine. A standard trick: use the Jacobian of the
# "solved" coord via the implicit function theorem.
#
# For Fermat quartic f = Σ z_i^4, df = 4 z_i^3 dz_i.
# Ω = dz_a ∧ dz_b / (4 z_c^3) where {a, b, c} omits the "solved" index.
# Squared norm: |Ω|² depends on z through the Jacobian.
#
# Loss σ² = Var_z [log det(g_K3(z)) - log |Ω(z)|²]
# This is the standard Donaldson measure of Ricci-flatness deviation.

def compute_omega_norm_sq(Z):
    """|Ω|² at each K3 point for the Fermat quartic.

    For f = Σ z_i^4 and Ω = (dz_a ∧ dz_b) / (4 z_c^3),
    the norm w.r.t. the induced metric on T_z K3 involves
    the pullback of the standard volume form.

    SIMPLIFIED: use |Ω|² ∝ 1/|∇f|²  (Poincaré residue normalization).
    """
    grad_f = 4.0 * (Z ** 3)  # (N, 4)
    grad_f_norm_sq = (grad_f.abs() ** 2).sum(dim=1)  # (N,) real
    # Up to an overall constant (absorbed in normalization),
    # |Ω|² ∝ 1 / |∇f|²
    omega_sq = 1.0 / grad_f_norm_sq
    return omega_sq  # (N,) real

def monge_ampere_loss(Z, H, mono_exp, k=4):
    """Donaldson loss: Var(log det g_K3 - log |Ω|²).

    Ricci-flat ↔ this is constant across z, i.e., Var = 0.
    """
    # Compute g on P^3 via autograd
    g_P3 = kahler_metric_real(Z, H, mono_exp, k)  # (N, 4, 4)

    # Pullback to K3 tangent
    basis, _ = compute_k3_tangent_basis(Z)  # (N, 4, 2)
    g_K3 = torch.einsum('nai,nab,nbj->nij', basis.conj(), g_P3, basis)  # (N, 2, 2)

    # log det g_K3
    det_K3 = torch.linalg.det(g_K3).real  # (N,) — real for Hermitian 2x2
    det_K3 = det_K3.clamp(min=1e-30)  # numerical stability
    log_det = torch.log(det_K3)

    # log |Ω|²
    omega_sq = compute_omega_norm_sq(Z)  # (N,)
    log_omega_sq = torch.log(omega_sq)

    # Monge-Ampère residual: log det g - log |Ω|² - c
    # (c is a constant we subtract to center the residual)
    residual = log_det - log_omega_sq
    c = residual.mean()
    residual_centered = residual - c

    # Loss: variance of residual
    sigma_sq = (residual_centered ** 2).mean()

    return sigma_sq, log_det, log_omega_sq, c

# Baseline: H = identity (Fubini-Study)
print('=== BASELINE: H = Identity (Fubini-Study) ===')
t0 = time.time()
sigma_sq_baseline, log_det, log_omega_sq, c_baseline = monge_ampere_loss(
    Z[:N_test], H_init, mono_exp, K_DEGREE
)
sigma_baseline = sigma_sq_baseline.sqrt().item()
print(f'  Time: {time.time()-t0:.2f}s')
print(f'  σ (Monge-Ampère variance) = {sigma_baseline:.6f}')
print(f'  log det g_K3 range: [{log_det.min().item():.4f}, {log_det.max().item():.4f}]')
print(f'  log |Ω|² range: [{log_omega_sq.min().item():.4f}, {log_omega_sq.max().item():.4f}]')
print(f'  Constant c = {c_baseline.item():.4f}')
print(f'  Residual σ: {sigma_baseline:.6f} (higher = further from Ricci-flat)')

print('\nDonaldson objective: reduce σ via optimization over H.')
print('Theorem (Donaldson): σ → 0 as k → ∞ (with increasing N_sections).')

=== BASELINE: H = Identity (Fubini-Study) ===
  Time: 0.55s
  σ (Monge-Ampère variance) = 0.535290
  log det g_K3 range: [-6.2680, -1.5124]
  log |Ω|² range: [-9.2628, -3.3728]
  Constant c = 2.1226
  Residual σ: 0.535290 (higher = further from Ricci-flat)

Donaldson objective: reduce σ via optimization over H.
Theorem (Donaldson): σ → 0 as k → ∞ (with increasing N_sections).


In [10]:
# ================================================================
# 9. DONALDSON OPTIMIZATION: LBFGS on H
# ================================================================
# Parametrize Hermitian H via its real/imaginary parts.
# N_SECT = 35, so H has 35² = 1225 real params (Hermitian has 35 real + 2*C(35,2) = 35 + 1190).

# Parametrize H = H_re + i H_im where H_re is symmetric, H_im is antisymmetric.
# Free params: upper triangle of H_re (including diagonal) + strict upper of H_im.
# Total: 35 + 35*34/2 + 35*34/2 = 35 + 595 + 595 = 1225 real params.

def hermitian_from_params(h_re, h_im, N):
    """Build Hermitian matrix from real/imag parametrizations."""
    H_re = (h_re + h_re.T) / 2  # symmetrize
    H_im = (h_im - h_im.T) / 2  # antisymmetrize
    # Zero out diagonal of H_im (pure imaginary diagonal = 0 for Hermitian)
    H_im = H_im - torch.diag(torch.diag(H_im))
    H = H_re.to(dtype) + 1j * H_im.to(dtype)
    return H

# Initialize from identity + small noise
torch.manual_seed(123)
h_re_param = torch.nn.Parameter(
    torch.eye(N_SECT, dtype=rdtype, device=device) + 0.01 * torch.randn(N_SECT, N_SECT, dtype=rdtype, device=device)
)
h_im_param = torch.nn.Parameter(
    0.01 * torch.randn(N_SECT, N_SECT, dtype=rdtype, device=device)
)

# Use more points for stable loss estimate
N_OPT = 500
Z_opt = Z[:N_OPT]

print(f'=== DONALDSON OPTIMIZATION ===')
print(f'N_params: {2 * N_SECT * N_SECT} real (symmetric + antisymmetric)')
print(f'N_SECT: {N_SECT}, N_opt points: {N_OPT}')

# LBFGS
optimizer = torch.optim.LBFGS(
    [h_re_param, h_im_param], lr=0.5, max_iter=20,
    line_search_fn='strong_wolfe', history_size=20
)

N_STEPS = 30
history = []
t0 = time.time()

def eval_sigma():
    H = hermitian_from_params(h_re_param, h_im_param, N_SECT)
    sigma_sq, _, _, _ = monge_ampere_loss(Z_opt, H, mono_exp, K_DEGREE)
    return sigma_sq

print(f'\n{"Step":>4s}  {"σ²":>12s}  {"σ":>12s}  {"time":>6s}')
print('-' * 40)

# Record baseline
with torch.no_grad():
    sigma_sq_0 = eval_sigma()
    sigma_0 = sigma_sq_0.sqrt().item()
print(f'{"init":>4s}  {sigma_sq_0.item():12.6e}  {sigma_0:12.6e}  {0.0:6.1f}s')

for step in range(N_STEPS):
    def closure():
        optimizer.zero_grad()
        loss = eval_sigma()
        loss.backward()
        return loss

    try:
        optimizer.step(closure)
    except RuntimeError as e:
        print(f'LBFGS error at step {step}: {e}')
        break

    with torch.no_grad():
        sigma_sq = eval_sigma()
        sigma = sigma_sq.sqrt().item()

    history.append({'step': step, 'sigma_sq': sigma_sq.item(), 'sigma': sigma, 'time': time.time()-t0})

    if step % 5 == 0 or step == N_STEPS - 1:
        print(f'{step:4d}  {sigma_sq.item():12.6e}  {sigma:12.6e}  {time.time()-t0:6.1f}s')

dt = time.time() - t0
reduction = sigma_0 / sigma if sigma > 0 else float('inf')
print(f'\n{"="*40}')
print(f'Baseline σ: {sigma_0:.6f}')
print(f'Final σ:    {sigma:.6f}')
print(f'Reduction:  {reduction:.2f}×')
print(f'Time:       {dt:.1f}s')

# Save optimized H
H_opt = hermitian_from_params(h_re_param, h_im_param, N_SECT).detach().cpu().numpy()
np.save(os.path.join(OUT_DIR, 'H_optimized.npy'), H_opt)

k3_results = {
    'K_degree': K_DEGREE,
    'N_sect': N_SECT,
    'N_points': N_OPT,
    'sigma_baseline': sigma_0,
    'sigma_final': sigma,
    'reduction': reduction,
    'n_steps': N_STEPS,
    'time_s': round(dt, 1),
    'history': history,
}
with open(os.path.join(OUT_DIR, 'k3_donaldson_results.json'), 'w') as f:
    json.dump(k3_results, f, indent=2, default=str)
print(f'\nSaved: {OUT_DIR}/k3_donaldson_results.json')

=== DONALDSON OPTIMIZATION ===
N_params: 2450 real (symmetric + antisymmetric)
N_SECT: 35, N_opt points: 500

Step            σ²             σ    time
----------------------------------------
init  2.745313e-01  5.239574e-01     0.0s
   0  1.451072e-03  3.809294e-02    45.9s
   5  1.740891e-05  4.172399e-03   199.5s
  10  2.535465e-06  1.592314e-03   352.0s
  15  6.457855e-07  8.036078e-04   504.8s
  20  2.028723e-07  4.504134e-04   657.2s
  25  1.344216e-07  3.666355e-04   729.9s
  29  1.182449e-07  3.438676e-04   757.6s

Baseline σ: 0.523957
Final σ:    0.000344
Reduction:  1523.72×
Time:       757.6s

Saved: /content/k3_cap_20260416T125431Z/k3_donaldson_results.json


In [11]:
# K-SWEEP: push Donaldson to k=6 and k=8
# Direct response to "thousands of params" objection.

k_sweep_results = {4: {
    'N_sect': N_SECT, 'n_params': 2 * N_SECT * N_SECT,
    'sigma_baseline': sigma_0, 'sigma_final': sigma,
    'reduction': reduction, 'n_steps': N_STEPS, 'time_s': round(dt, 1),
}}

def run_donaldson_at_k(k_new, n_opt=1000, n_steps=25, n_sample=2000):
    print(f'\n=== DONALDSON k={k_new} ===')
    monos_new = generate_monomials(k_new, 4)
    N_new = len(monos_new)
    mono_new = torch.tensor(monos_new, dtype=torch.long, device=device)
    n_p = 2 * N_new * N_new
    print(f'  N_sections={N_new}, params={n_p}')

    Z_k = sample_k3_points(n_sample)
    Z_opt_k = Z_k[:n_opt]
    basis_k, _ = compute_k3_tangent_basis(Z_opt_k)

    def ma_loss(Z_pts, H_mat, basis_pts):
        X = Z_pts.real.detach().clone().requires_grad_(True)
        Y = Z_pts.imag.detach().clone().requires_grad_(True)
        from torch.func import jacrev, vmap
        def K_single(x, y):
            z = (x + 1j*y).to(dtype)
            s = torch.stack([torch.prod(z**mono_new[a].to(dtype)) for a in range(N_new)])
            phi = (s.conj() @ H_mat @ s).real
            return torch.log(phi) / k_new
        K_xx = vmap(jacrev(jacrev(K_single, argnums=0), argnums=0))(X, Y)
        K_yy = vmap(jacrev(jacrev(K_single, argnums=1), argnums=1))(X, Y)
        K_xy = vmap(jacrev(jacrev(K_single, argnums=0), argnums=1))(X, Y)
        g_P3 = 0.25*(K_xx + K_yy) + 1j*0.25*(K_xy.transpose(1,2) - K_xy)
        g_k3 = torch.einsum('nai,nab,nbj->nij', basis_pts.conj(), g_P3, basis_pts)
        det_k3 = torch.linalg.det(g_k3).real.clamp(min=1e-30)
        omega_sq = compute_omega_norm_sq(Z_pts)
        residual = torch.log(det_k3) - torch.log(omega_sq)
        return ((residual - residual.mean()) ** 2).mean()

    torch.manual_seed(42 + k_new)
    h_re = torch.nn.Parameter(torch.eye(N_new, dtype=rdtype, device=device) +
                              0.01*torch.randn(N_new, N_new, dtype=rdtype, device=device))
    h_im = torch.nn.Parameter(0.01*torch.randn(N_new, N_new, dtype=rdtype, device=device))
    opt = torch.optim.LBFGS([h_re, h_im], lr=0.5, max_iter=15,
                             line_search_fn='strong_wolfe', history_size=15)

    t0 = time.time()
    hist = []
    with torch.no_grad():
        H = hermitian_from_params(h_re, h_im, N_new)
        sig_init = math.sqrt(ma_loss(Z_opt_k, H, basis_k).item())
    print(f'  init: sigma={sig_init:.4e}')

    for step in range(n_steps):
        def closure():
            opt.zero_grad()
            H = hermitian_from_params(h_re, h_im, N_new)
            loss = ma_loss(Z_opt_k, H, basis_k)
            loss.backward()
            return loss
        try:
            opt.step(closure)
        except RuntimeError as e:
            print(f'  err: {e}'); break
        with torch.no_grad():
            H = hermitian_from_params(h_re, h_im, N_new)
            sig = math.sqrt(ma_loss(Z_opt_k, H, basis_k).item())
        hist.append({'step': step, 'sigma': sig, 'time': time.time()-t0})
        if step % 5 == 0 or step == n_steps - 1:
            print(f'  step {step}: sigma={sig:.4e}  ({time.time()-t0:.1f}s)')

    dt_k = time.time() - t0
    sig_f = hist[-1]['sigma'] if hist else sig_init
    red = sig_init / sig_f if sig_f > 0 else float('inf')
    print(f'\n  k={k_new}: {sig_init:.3e} -> {sig_f:.3e} ({red:.1f}x) in {dt_k:.0f}s')

    H_out = hermitian_from_params(h_re, h_im, N_new).detach().cpu().numpy()
    np.save(os.path.join(OUT_DIR, f'H_k{k_new}_optimized.npy'), H_out)

    return {
        'k': k_new, 'N_sect': N_new, 'n_params': n_p,
        'sigma_baseline': sig_init, 'sigma_final': sig_f,
        'reduction': red, 'n_steps': n_steps, 'time_s': round(dt_k, 1),
        'history': hist,
    }

for k_target in [6, 8]:
    k_sweep_results[k_target] = run_donaldson_at_k(k_target, n_opt=1000, n_steps=25, n_sample=2000)

print(f'\n{"="*70}\nK-SWEEP SUMMARY\n{"="*70}')
print(f'{"k":>3s}  {"sections":>8s}  {"params":>8s}  '
      f'{"sigma_init":>12s}  {"sigma_final":>12s}  {"reduction":>10s}  {"time":>7s}')
print('-' * 70)
for k in sorted(k_sweep_results.keys()):
    r = k_sweep_results[k]
    print(f'{k:3d}  {str(r.get("N_sect","?")):>8s}  {r["n_params"]:8d}  '
          f'{r["sigma_baseline"]:12.3e}  {r["sigma_final"]:12.3e}  '
          f'{r["reduction"]:10.1f}x  {r["time_s"]:6.1f}s')

with open(os.path.join(OUT_DIR, 'k_sweep_results.json'), 'w') as f:
    json.dump(k_sweep_results, f, indent=2, default=str)
print(f'\nSaved to {OUT_DIR}/k_sweep_results.json')


=== DONALDSON k=6 ===
  N_sections=84, params=14112
  Sampled 2000 K3 points
  Max constraint residual |f(z)| = 8.67e-14
  init: sigma=6.1702e-01
  step 0: sigma=6.0347e-02  (63.0s)
  step 5: sigma=8.5292e-03  (345.0s)
  step 10: sigma=3.9337e-03  (632.2s)
  step 15: sigma=2.2949e-03  (914.9s)
  step 20: sigma=1.5907e-03  (1198.2s)
  step 24: sigma=1.2964e-03  (1424.8s)

  k=6: 6.170e-01 -> 1.296e-03 (475.9x) in 1425s

=== DONALDSON k=8 ===
  N_sections=165, params=54450
  Sampled 2000 K3 points
  Max constraint residual |f(z)| = 3.98e-14
  init: sigma=6.9666e-01
  step 0: sigma=1.6637e-01  (126.0s)
  step 5: sigma=9.1895e-03  (678.9s)
  step 10: sigma=4.0555e-03  (1229.4s)
  step 15: sigma=2.3413e-03  (1779.9s)
  step 20: sigma=1.5250e-03  (2337.9s)
  step 24: sigma=1.1363e-03  (2780.1s)

  k=8: 6.967e-01 -> 1.136e-03 (613.1x) in 2780s

K-SWEEP SUMMARY
  k  sections    params    sigma_init   sigma_final   reduction     time
------------------------------------------------------------

In [15]:
# ================================================================
  # K=8 PROPER RUN — "tens of thousands of params" (Platt, 2026-04-15)
  # ================================================================
  # Previous k=8 run: n_opt=1000, n_steps=25 → σ=1.14e-3 (undertrained)
  # This run: n_opt=1000, n_steps=80, n_sample=3000 → ~90-120 min on A100

result_k8_v2 = run_donaldson_at_k(
      k_new=8,
      n_opt=1000,    # same as before (bottleneck is steps, not data)
      n_steps=80,    # ×3.2 — this is the key lever
      n_sample=3000, # slightly more diverse sampling
  )

print(f"\n{'='*60}")
print(f"K=8 FINAL:  σ = {result_k8_v2['sigma_final']:.4e}")
print(f"N_params  = {result_k8_v2['n_params']:,}  (54,450 for k=8)")
print(f"Reduction = {result_k8_v2['reduction']:.1f}×")
print(f"{'='*60}")

  # Save explicitly
np.save(os.path.join(OUT_DIR, 'H_k8_v2_optimized.npy'),
          # rebuild H from current h_re/h_im state — need to recover from run_donaldson_at_k
          # (function saves internally to OUT_DIR/H_k8_optimized.npy)
          np.load(os.path.join(OUT_DIR, 'H_k8_optimized.npy')))


=== DONALDSON k=8 ===
  N_sections=165, params=54450
  Sampled 3000 K3 points
  Max constraint residual |f(z)| = 6.90e-14
  init: sigma=7.2905e-01
  step 0: sigma=1.2162e-01  (126.3s)
  step 5: sigma=1.0540e-02  (682.2s)
  step 10: sigma=4.3895e-03  (1258.8s)
  step 15: sigma=2.3356e-03  (1824.1s)
  step 20: sigma=1.4529e-03  (2367.7s)
  step 25: sigma=9.7719e-04  (2911.6s)
  step 30: sigma=6.6255e-04  (3467.2s)
  step 35: sigma=4.5141e-04  (4031.0s)
  step 40: sigma=4.0239e-04  (4293.4s)
  step 45: sigma=3.9113e-04  (4389.5s)
  step 50: sigma=3.7316e-04  (4518.7s)
  step 55: sigma=3.5485e-04  (4627.5s)
  step 60: sigma=3.4522e-04  (4716.3s)
  step 65: sigma=3.3541e-04  (4812.2s)
  step 70: sigma=3.2732e-04  (4901.0s)
  step 75: sigma=3.2111e-04  (4976.7s)
  step 79: sigma=3.1264e-04  (5057.1s)

  k=8: 7.291e-01 -> 3.126e-04 (2332.0x) in 5057s

K=8 FINAL:  σ = 3.1264e-04
N_params  = 54,450  (54,450 for k=8)
Reduction = 2332.0×


In [25]:
# Fix NK cert pour k=8 — N_SECT est capturé globalement par K_single
N_SECT = N_k8   # 165, override le 35 global

with torch.no_grad():
      g_P3_k8 = kahler_metric_real(Z_ev, H_k8_t, mono_k8_t, k=8)
      g_K3_k8 = torch.einsum('nai,nab,nbj->nij',
                              basis_ev.conj(), g_P3_k8, basis_ev)
      det_k8     = torch.linalg.det(g_K3_k8).real.clamp(min=1e-30)
      residual_k8 = torch.log(det_k8) - torch.log(compute_omega_norm_sq(Z_ev))
      residual_k8 = residual_k8 - residual_k8.mean()
      eigs_k8    = torch.linalg.eigvalsh(g_K3_k8)
      lam_min_k8 = eigs_k8.min().item()

N_SECT = 35   # restore k=4 global

eta_L2_k8  = residual_k8.pow(2).mean().sqrt().item()
eta_sup_k8 = residual_k8.abs().max().item()
omega_k8   = (1.0 / eigs_k8.flatten().median().item()) ** 2  # L² variant
beta       = 0.1

h_L2_k8 = beta * eta_L2_k8 * omega_k8
print(f'K=8 NK L²: h = {h_L2_k8:.4f}  →  {"PASS ×"+str(round(0.5/h_L2_k8)) if h_L2_k8 < 0.5 else "FAIL"}')
print(f'  σ={3.1264e-4:.4e}, η_L²={eta_L2_k8:.3e}, ω_L2={omega_k8:.1f}')

K=8 NK L²: h = 0.2124  →  PASS ×2
  σ=3.1264e-04, η_L²=1.375e-01, ω_L2=15.4
